# KA-5–KA-6: bounded workflow and durable recovery

Implementation notes are English; bilingual teaching text lives in the chapters and ka5-content-v1.json. Prerequisites: Python Primer, HTTP Primer and Chapters 40–49. Run every cell on CPU. The callback is explicitly authored; saved real-model evidence is inspected separately. Apache-2.0 code; original prose CC BY-SA 4.0.

In [1]:
from pathlib import Path
import sys, json, tempfile
ROOT = Path.cwd()
while not (ROOT / 'src/config/book.mjs').exists():
    if ROOT == ROOT.parent: raise RuntimeError('repository_not_found')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'code/knowledge-assistant'))
sys.path.insert(0, str(ROOT / 'code/part-viii'))
from contracts import load
from workflow import run
from reliable_runtime import Principal, Store, action, retry_schedule
from source_skill import load_trace
from security_controls import verify_inventory
inventory = load('../part-viii/reviewed-inventory-v1.json')
assert verify_inventory(ROOT, inventory['files'])['accepted']
print('Reviewed inputs verified; no model service is contacted.')

Reviewed inputs verified; no model service is contacted.


The independent expected state paths were hand-enumerated before implementation. Empty evidence must avoid the decision call.

In [2]:
expected = load('../part-viii/expected-v1.json')
for locale in ['en', 'zh-hans']:
    question = load('ka5-content-v1.json')['locales'][locale]['question']
    result = run(question, locale=locale, choose=lambda _: 'use_evidence')
    assert [event['state'] for event in result['events']] == expected['workflow']['valid']
    assert result['answer']['amount_yuan'] == 750
    print(locale, result['state'], result['model_calls'], result['answer']['amount_yuan'])
empty = run('Beijing lodging', documents=[], choose=lambda _: (_ for _ in ()).throw(AssertionError('unexpected model call')))
assert empty['state'] == 'ABSTAINED' and empty['model_calls'] == 0
print('Empty evidence:', empty['state'], empty['model_calls'])

en ANSWERED 1 750
zh-hans ANSWERED 1 750
Empty evidence: ABSTAINED 0


In [3]:
result = run('Beijing lodging', choose=lambda _: 'use_evidence')
prefix = 'code/knowledge-assistant/skills/source-verification/'
package_hashes = {row['path'][len(prefix):]: row['sha256'] for row in inventory['files'] if row['path'].startswith(prefix)}
trace = load_trace(dict(context=result['retrieval']['context'], answer=result['answer']), package_hashes)
assert trace[-1]['report']['supported']
print(json.dumps(trace, indent=2))

[
  {
    "stage": "discover",
    "loaded_fields": {
      "name": "source-verification",
      "description": "Verify dated travel-policy evidence and exact citation spans in the Understanding LLMs knowledge assistant. Use for checking a proposed answer against the local fictional corpus."
    },
    "utf8_bytes": 225
  },
  {
    "stage": "activate",
    "path": "SKILL.md",
    "utf8_bytes": 1511
  },
  {
    "stage": "reference",
    "path": "references/evidence-contract.md",
    "utf8_bytes": 1127
  },
  {
    "stage": "execute",
    "path": "scripts/verify.py",
    "returncode": 0,
    "report": {
      "schema_version": "source-verification-report-v1",
      "supported": true,
      "authority_ok": true,
      "context_matches": true,
      "reason": "supported",
      "checks": {
        "citations": [
          {
            "exists": true,
            "exact": true,
            "eligible": true,
            "complete_scope": true,
            "amount_matches": true,
         

A deliberately lost reply occurs after the local transaction commits. Reopen the database, resume with the same key, and compare actual effects and replay mutations.

In [4]:
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / 'orders.sqlite'
    user = Principal('north', 'mira', scopes=('orders:read:own', 'orders:cancel:own'))
    reviewer = Principal('north', 'reviewer', 'approver', scopes=('orders:approve',))
    store = Store(path, clock=lambda: 1000.0)
    approval = store.approve(reviewer, user, action())
    store.start(user, 'request', action(), approval, 'one-intent')
    first = store.advance(user, 'request', fault='after_commit')
    assert first['state'] == 'UNCERTAIN'
    store.close()
    store = Store(path, clock=lambda: 1000.0)
    resumed = store.advance(user, 'request')
    before = store.db.total_changes
    events = store.replay(user, 'request')
    assert store.db.total_changes == before
    assert store.read(user, 'A-104')['effects'] == 1
    assert resumed['result']['receipt_reused']
    print('Resumed:', resumed['state'], 'remaining:', resumed['remaining'], 'effects: 1; replay mutations: 0')
    print(json.dumps(events, indent=2))
    store.close()
assert retry_schedule()['elapsed_seconds'] == 0.35
assert len(retry_schedule(deadline=0.21)['events']) == 1
print('Logical time:', retry_schedule())

Resumed: DONE remaining: 1 effects: 1; replay mutations: 0
[
  {
    "sequence": 1,
    "at": 1000.0,
    "state": "PENDING",
    "detail": {
      "action_hash": "f7117e2dc7d252ebc11251f2468aba742be3edd99efecc605aec0c3434dc1c50",
      "budget": 3
    }
  },
  {
    "sequence": 2,
    "at": 1000.0,
    "state": "CALLING",
    "detail": {
      "attempt": 1,
      "remaining": 2
    }
  },
  {
    "sequence": 3,
    "at": 1000.0,
    "state": "UNCERTAIN",
    "detail": {
      "reason": "response_lost_after_commit"
    }
  },
  {
    "sequence": 4,
    "at": 1000.0,
    "state": "CALLING",
    "detail": {
      "attempt": 2,
      "remaining": 1
    }
  },
  {
    "sequence": 5,
    "at": 1000.0,
    "state": "DONE",
    "detail": {
      "status": "cancelled",
      "order_id": "A-104",
      "version": 2,
      "receipt_reused": true
    }
  }
]
Logical time: {'events': [{'attempt': 1, 'budget': 2, 'elapsed_seconds': 0.2, 'status': 'timeout'}, {'attempt': 2, 'budget': 1, 'elapsed_sec

In [5]:
from permission_records import run as permissions
matrix = permissions()
allowed = [row['case'] for row in matrix['rows'] if row['allowed']]
assert allowed == ['own_read', 'south_own_read', 'approved_cancel']
print(json.dumps(matrix, indent=2))
model = load('../part-viii/model-run.json')
print('Saved optional-model record:', model.keys())

{
  "provenance": "Actual local matrix; fresh SQLite database per row; host principals and approvals are synthetic, not production credentials.",
  "rows": [
    {
      "case": "own_read",
      "allowed": true,
      "result": {
        "tenant": "north",
        "id": "A-104",
        "owner": "mira",
        "status": "processing",
        "version": 1,
        "effects": 0
      },
      "north_effects": 0
    },
    {
      "case": "south_own_read",
      "allowed": true,
      "result": {
        "tenant": "south",
        "id": "A-104",
        "owner": "leo",
        "status": "shipped",
        "version": 1,
        "effects": 0
      },
      "north_effects": 0
    },
    {
      "case": "wrong_owner",
      "allowed": false,
      "result": {
        "reason": "not_found_or_forbidden"
      },
      "north_effects": 0
    },
    {
      "case": "wrong_tenant",
      "allowed": false,
      "result": {
        "reason": "not_found_or_forbidden"
      },
      "north_effects"